In [1]:
import os
import re
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# =========================
# PATHS
# =========================
CANDIDATE_BASE_DIRS = [
    Path("/mnt/e/CVPR"),
    Path("/mnt/d/CVPR"),
    Path(r"E:\CVPR"),
    Path(r"D:\CVPR"),
]

BASE_DIR = None
for p in CANDIDATE_BASE_DIRS:
    if p.exists():
        BASE_DIR = p
        break

if BASE_DIR is None:
    raise FileNotFoundError("Could not find CVPR folder.")

PROJECT_ROOT = BASE_DIR / "steel_failure_aware"
RUNS_DIR = PROJECT_ROOT / "runs"

CLOSED_SET_DIR = RUNS_DIR / "closed_set"
CALIBRATION_DIR = RUNS_DIR / "calibration"
SELECTIVE_DIR = RUNS_DIR / "selective_prediction"
EXTERNAL_NEU_DIR = RUNS_DIR / "external_unknown_neu"
LOCO_DIR = RUNS_DIR / "internal_unknown_gc10_loco"

FINAL_DIR = PROJECT_ROOT / "final_tables_and_figures"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

TABLES_DIR = FINAL_DIR / "tables"
FIGURES_DIR = FINAL_DIR / "figures"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("CLOSED_SET   :", CLOSED_SET_DIR)
print("CALIBRATION  :", CALIBRATION_DIR)
print("SELECTIVE    :", SELECTIVE_DIR)
print("EXTERNAL_NEU :", EXTERNAL_NEU_DIR)
print("LOCO         :", LOCO_DIR)
print("FINAL_DIR    :", FINAL_DIR)

PROJECT_ROOT : /mnt/e/CVPR/steel_failure_aware
CLOSED_SET   : /mnt/e/CVPR/steel_failure_aware/runs/closed_set
CALIBRATION  : /mnt/e/CVPR/steel_failure_aware/runs/calibration
SELECTIVE    : /mnt/e/CVPR/steel_failure_aware/runs/selective_prediction
EXTERNAL_NEU : /mnt/e/CVPR/steel_failure_aware/runs/external_unknown_neu
LOCO         : /mnt/e/CVPR/steel_failure_aware/runs/internal_unknown_gc10_loco
FINAL_DIR    : /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures


In [2]:
def pretty_model_name(model_name: str) -> str:
    mapping = {
        "mobilenetv3_small": "MobileNetV3-Small",
        "resnet18": "ResNet-18",
        "customcnn_bn": "CustomCNN-BN",
        "customcnn_drop": "CustomCNN-Drop",
        "customcnn_base": "CustomCNN-Base",
    }
    return mapping.get(model_name, model_name)

def parse_run_name(run_name: str) -> dict:
    """
    Parse strings like:
    mobilenetv3_small__adamw__weightedCE_True__labelsmooth_True__pretrained_True__seed_42
    or
    mobilenetv3_small__adamw__weightedCE_True__labelsmooth_True__pretrained_True__seed_42__score_raw_msp
    """
    parts = run_name.split("__")
    out = {
        "run_name": run_name,
        "model_raw": None,
        "model": None,
        "optimizer": None,
        "weightedCE": None,
        "labelsmooth": None,
        "pretrained": None,
        "seed": None,
        "score": None,
    }

    if len(parts) >= 1:
        out["model_raw"] = parts[0]
        out["model"] = pretty_model_name(parts[0])

    if len(parts) >= 2:
        out["optimizer"] = parts[1]

    for p in parts:
        if p.startswith("weightedCE_"):
            out["weightedCE"] = p.replace("weightedCE_", "")
        elif p.startswith("labelsmooth_"):
            out["labelsmooth"] = p.replace("labelsmooth_", "")
        elif p.startswith("pretrained_"):
            out["pretrained"] = p.replace("pretrained_", "")
        elif p.startswith("seed_"):
            seed_str = p.replace("seed_", "")
            if seed_str.isdigit():
                out["seed"] = int(seed_str)

    if "__score_" in run_name:
        out["score"] = run_name.split("__score_")[-1]

    return out

def first_existing_file(candidates):
    for p in candidates:
        if p.exists():
            return p
    return None

def safe_read_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_json_or_csv_dict(path: Path):
    if path.suffix.lower() == ".json":
        return safe_read_json(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        if len(df) == 0:
            return {}
        return df.iloc[0].to_dict()
    else:
        return {}

def normalize_number(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def find_column(d: dict, candidates, default=np.nan):
    for c in candidates:
        if c in d:
            return d[c]
    return default

def copy_if_exists(src: Path, dst: Path):
    if src is not None and src.exists():
        shutil.copy2(src, dst)
        return True
    return False

print("Helper functions ready.")

Helper functions ready.


In [3]:
closed_rows = []

for run_dir in CLOSED_SET_DIR.iterdir():
    if not run_dir.is_dir():
        continue

    info = parse_run_name(run_dir.name)

    metrics_json = run_dir / "metrics_test.json"
    config_json = run_dir / "config.json"
    run_summary_csv = run_dir / "run_summary.csv"

    row = {
        "RunName": run_dir.name,
        "Model": info["model"],
        "Seed": info["seed"],
        "Optimizer": info["optimizer"],
        "WeightedCE": info["weightedCE"],
        "LabelSmoothing": info["labelsmooth"],
        "Pretrained": info["pretrained"],
    }

    if metrics_json.exists():
        mj = safe_read_json(metrics_json)
        row.update({
            "BestEpoch": mj.get("best_epoch", np.nan),
            "BestValMacroF1": mj.get("best_val_macro_f1", np.nan),
            "TestLoss": mj.get("test_loss", np.nan),
            "Accuracy": mj.get("accuracy", np.nan),
            "MacroPrecision": mj.get("macro_precision", np.nan),
            "MacroRecall": mj.get("macro_recall", np.nan),
            "MacroF1": mj.get("macro_f1", np.nan),
            "WeightedPrecision": mj.get("weighted_precision", np.nan),
            "WeightedRecall": mj.get("weighted_recall", np.nan),
            "WeightedF1": mj.get("weighted_f1", np.nan),
        })

    if config_json.exists():
        cj = safe_read_json(config_json)
        row.update({
            "ImageSize": cj.get("image_size", np.nan),
            "BatchSize": cj.get("batch_size", np.nan),
            "LearningRate": cj.get("learning_rate", np.nan),
            "WeightDecay": cj.get("weight_decay", np.nan),
            "NumEpochsConfigured": cj.get("num_epochs", np.nan),
        })

    if run_summary_csv.exists():
        rs = pd.read_csv(run_summary_csv)
        if len(rs) > 0:
            rs0 = rs.iloc[0].to_dict()
            for k, v in rs0.items():
                if k not in row:
                    row[k] = v

    if "Accuracy" in row and not pd.isna(row["Accuracy"]):
        closed_rows.append(row)

closed_df = pd.DataFrame(closed_rows)
closed_df = closed_df.sort_values(by=["MacroF1", "Accuracy"], ascending=[False, False]).reset_index(drop=True)

closed_csv = TABLES_DIR / "closed_set_final_benchmark_table.csv"
closed_xlsx = TABLES_DIR / "closed_set_final_benchmark_table.xlsx"
closed_df.to_csv(closed_csv, index=False)
closed_df.to_excel(closed_xlsx, index=False)

closed_grouped = (
    closed_df.groupby("Model", as_index=False)
    .agg(
        Runs=("RunName", "count"),
        AccuracyMean=("Accuracy", "mean"),
        AccuracyStd=("Accuracy", "std"),
        MacroF1Mean=("MacroF1", "mean"),
        MacroF1Std=("MacroF1", "std"),
        WeightedF1Mean=("WeightedF1", "mean"),
        WeightedF1Std=("WeightedF1", "std"),
        BestValMacroF1Mean=("BestValMacroF1", "mean"),
        BestValMacroF1Std=("BestValMacroF1", "std"),
    )
    .sort_values(by="MacroF1Mean", ascending=False)
    .reset_index(drop=True)
)

closed_grouped_csv = TABLES_DIR / "closed_set_grouped_mean_std_table.csv"
closed_grouped_xlsx = TABLES_DIR / "closed_set_grouped_mean_std_table.xlsx"
closed_grouped.to_csv(closed_grouped_csv, index=False)
closed_grouped.to_excel(closed_grouped_xlsx, index=False)

print("Saved:", closed_csv)
print("Saved:", closed_xlsx)
print("Saved:", closed_grouped_csv)
print("Saved:", closed_grouped_xlsx)

display(closed_df)
display(closed_grouped)

Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/closed_set_final_benchmark_table.csv
Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/closed_set_final_benchmark_table.xlsx
Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/closed_set_grouped_mean_std_table.csv
Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/closed_set_grouped_mean_std_table.xlsx


,RunName,Model,Seed,Optimizer,WeightedCE,LabelSmoothing,Pretrained,BestEpoch,BestValMacroF1,TestLoss,Accuracy,MacroPrecision,MacroRecall,MacroF1,WeightedPrecision,WeightedRecall,WeightedF1,ImageSize,BatchSize,LearningRate,WeightDecay,NumEpochsConfigured,run_name,seed,model_name,optimizer_name,best_epoch,best_val_macro_f1,test_loss,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
0,resnet18__adamw__weightedCE_True__labelsmooth_...,ResNet-18,87,adamw,True,True,True,15,0.843762,0.916700,0.887608,0.836072,0.835889,0.833037,0.892648,0.887608,0.887325,224,32,0.0001,0.0005,30,resnet18__adamw__weightedCE_True__labelsmooth_...,87,resnet18,adamw,15,0.843762,0.916700,0.887608,0.836072,0.835889,0.833037,0.892648,0.887608,0.887325
1,resnet18__adamw__weightedCE_True__labelsmooth_...,ResNet-18,13,adamw,True,True,True,12,0.839280,0.861169,0.887608,0.820849,0.852782,0.832623,0.893805,0.887608,0.889945,224,32,0.0001,0.0005,30,resnet18__adamw__weightedCE_True__labelsmooth_...,13,resnet18,adamw,12,0.839280,0.861169,0.887608,0.820849,0.852782,0.832623,0.893805,0.887608,0.889945
2,mobilenetv3_small__adamw__weightedCE_True__lab...,MobileNetV3-Small,42,adamw,True,True,True,13,0.785566,0.999508,0.844380,0.820675,0.815456,0.801657,0.862397,0.844380,0.849765,224,32,0.0001,0.0005,30,mobilenetv3_small__adamw__weightedCE_True__lab...,42,mobilenetv3_small,adamw,13,0.785566,0.999508,0.844380,0.820675,0.815456,0.801657,0.862397,0.844380,0.849765
3,mobilenetv3_small__adamw__weightedCE_True__lab...,MobileNetV3-Small,13,adamw,True,True,True,23,0.791644,0.967768,0.876081,0.785273,0.795671,0.787633,0.878524,0.876081,0.875187,224,32,0.0001,0.0005,30,mobilenetv3_small__adamw__weightedCE_True__lab...,13,mobilenetv3_small,adamw,23,0.791644,0.967768,0.876081,0.785273,0.795671,0.787633,0.878524,0.876081,0.875187
4,mobilenetv3_small__adamw__weightedCE_True__lab...,MobileNetV3-Small,87,adamw,True,True,True,19,0.783316,1.062584,0.847262,0.773458,0.791306,0.773547,0.853169,0.847262,0.846152,224,32,0.0001,0.0005,30,mobilenetv3_small__adamw__weightedCE_True__lab...,87,mobilenetv3_small,adamw,19,0.783316,1.062584,0.847262,0.773458,0.791306,0.773547,0.853169,0.847262,0.846152
5,resnet18__adamw__weightedCE_True__labelsmooth_...,ResNet-18,42,adamw,True,True,True,27,0.684541,1.340839,0.746398,0.669514,0.662200,0.645067,0.781597,0.746398,0.743585,224,32,0.0001,0.0005,30,resnet18__adamw__weightedCE_True__labelsmooth_...,42,resnet18,adamw,27,0.684541,1.340839,0.746398,0.669514,0.662200,0.645067,0.781597,0.746398,0.743585
6,resnet18__adamw__weightedCE__pretrained_True__...,ResNet-18,42,adamw,None,None,True,24,0.663880,1.433271,0.694524,0.630972,0.622371,0.601574,0.729861,0.694524,0.686880,224,32,0.0003,0.0001,30,resnet18__adamw__weightedCE__pretrained_True__...,42,resnet18,adamw,24,0.663880,1.433271,0.694524,0.630972,0.622371,0.601574,0.729861,0.694524,0.686880
7,customcnn_bn__adamw__weightedCE_True__labelsmo...,CustomCNN-BN,42,adamw,True,False,False,36,0.596589,1.236074,0.668588,0.573494,0.586145,0.564038,0.692974,0.668588,0.666429,224,32,0.0010,0.0001,40,customcnn_bn__adamw__weightedCE_True__labelsmo...,42,customcnn_bn,adamw,36,0.596589,1.236074,0.668588,0.573494,0.586145,0.564038,0.692974,0.668588,0.666429
8,customcnn_drop__adamw__weightedCE_True__labels...,CustomCNN-Drop,42,adamw,True,False,False,33,0.385959,1.368961,0.484150,0.452658,0.502506,0.416588,0.591904,0.484150,0.479407,224,32,0.0010,0.0001,40,customcnn_drop__adamw__weightedCE_True__labels...,42,customcnn_drop,adamw,33,0.385959,1.368961,0.484150,0.452658,0.502506,0.416588,0.591904,0.484150,0.479407
9,customcnn_base__adamw__seed_42,CustomCNN-Base,42,adamw,None,None,None,16,0.136124,1.869125,0.250720,0.184477,0.172203,0.142206,0.222072,0.250720,0.198518,224,32,0.0010,0.0001,40,customcnn_base__adamw__seed_42,42,customcnn_base,adamw,16,0.136124,1.869125,0.250720,0.184477,0.172203,0.142206,0.222072,0.250720,0.198518


,Model,Runs,AccuracyMean,AccuracyStd,MacroF1Mean,MacroF1Std,WeightedF1Mean,WeightedF1Std,BestValMacroF1Mean,BestValMacroF1Std
0,MobileNetV3-Small,3,0.855908,0.017530,0.787612,0.014055,0.857035,0.015824,0.786842,0.004308
1,ResNet-18,4,0.804035,0.098799,0.728075,0.122257,0.801934,0.102761,0.757866,0.096981
2,CustomCNN-BN,1,0.668588,NaN,0.564038,NaN,0.666429,NaN,0.596589,NaN
3,CustomCNN-Drop,1,0.484150,NaN,0.416588,NaN,0.479407,NaN,0.385959,NaN
4,CustomCNN-Base,1,0.250720,NaN,0.142206,NaN,0.198518,NaN,0.136124,NaN


In [11]:
# =========================
# Cell 4 (REPLACEMENT)
# Robust calibration table builder
# =========================
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ---------- fallback parse_run_name ----------
def _pretty_model_name(model_name: str) -> str:
    mapping = {
        "mobilenetv3_small": "MobileNetV3-Small",
        "resnet18": "ResNet-18",
        "customcnn_bn": "CustomCNN-BN",
        "customcnn_drop": "CustomCNN-Drop",
        "customcnn_base": "CustomCNN-Base",
    }
    return mapping.get(model_name, model_name)

def _parse_run_name_local(run_name: str) -> dict:
    parts = run_name.split("__")
    out = {
        "run_name": run_name,
        "model_raw": parts[0] if len(parts) > 0 else None,
        "model": _pretty_model_name(parts[0]) if len(parts) > 0 else None,
        "optimizer": parts[1] if len(parts) > 1 else None,
        "seed": None,
        "score": None,
    }
    for p in parts:
        if p.startswith("seed_"):
            s = p.replace("seed_", "")
            if s.isdigit():
                out["seed"] = int(s)
        if p.startswith("weightedCE_"):
            out["weightedCE"] = p.replace("weightedCE_", "")
        if p.startswith("labelsmooth_"):
            out["labelsmooth"] = p.replace("labelsmooth_", "")
        if p.startswith("pretrained_"):
            out["pretrained"] = p.replace("pretrained_", "")
    if "__score_" in run_name:
        out["score"] = run_name.split("__score_")[-1]
    return out

# ---------- helpers ----------
def _canon(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).strip().lower()).strip("_")

def _flatten_dict(d, parent_key=""):
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}_{k}" if parent_key else str(k)
        if isinstance(v, dict):
            items.update(_flatten_dict(v, new_key))
        else:
            items[new_key] = v
    return items

def _split_from_text(text):
    t = _canon(text)
    if any(x in t for x in ["validation", "val"]):
        return "Val"
    if any(x in t for x in ["test"]):
        return "Test"
    return None

def _cal_from_text(text):
    t = _canon(text)
    if any(x in t for x in ["uncal", "uncalibrated", "raw", "before_calibration"]):
        return "uncal"
    if any(x in t for x in ["calibrated", "after_calibration"]) and "uncal" not in t:
        return "cal"
    return None

def _metric_from_text(text):
    t = _canon(text)
    if "nll" in t:
        return "NLL"
    if "brier" in t:
        return "Brier"
    if "ece" in t:
        return "ECE"
    if "mce" in t:
        return "MCE"
    if t in ["temperature", "temp", "best_temperature", "optimal_temperature"]:
        return "Temperature"
    return None

def _safe_float(x):
    try:
        if pd.isna(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def _empty_calibration_row():
    return {
        "Val NLL uncal": np.nan,
        "Val NLL cal": np.nan,
        "Val Brier uncal": np.nan,
        "Val Brier cal": np.nan,
        "Val ECE uncal": np.nan,
        "Val ECE cal": np.nan,
        "Val MCE uncal": np.nan,
        "Val MCE cal": np.nan,
        "Test NLL uncal": np.nan,
        "Test NLL cal": np.nan,
        "Test Brier uncal": np.nan,
        "Test Brier cal": np.nan,
        "Test ECE uncal": np.nan,
        "Test ECE cal": np.nan,
        "Test MCE uncal": np.nan,
        "Test MCE cal": np.nan,
        "Temperature": np.nan,
    }

def _update_from_key_value(row, key, value):
    key_c = _canon(key)
    metric = _metric_from_text(key_c)

    if metric == "Temperature":
        val = _safe_float(value)
        if not pd.isna(val):
            row["Temperature"] = val
        return

    split = _split_from_text(key_c)
    cal_state = _cal_from_text(key_c)

    if split is not None and cal_state is not None and metric is not None:
        col = f"{split} {metric} {cal_state}"
        if col in row:
            val = _safe_float(value)
            if not pd.isna(val):
                row[col] = val

def _extract_from_flat_dict(flat_d):
    row = _empty_calibration_row()

    # pass 1: direct key parsing
    for k, v in flat_d.items():
        _update_from_key_value(row, k, v)

    # pass 2: handle possible three-column long style already flattened strangely
    return row

def _extract_from_dataframe(df):
    row = _empty_calibration_row()
    if df is None or len(df) == 0:
        return row

    # normalize columns
    df = df.copy()
    df.columns = [_canon(c) for c in df.columns]

    # -------------------------
    # Case A: wide format
    # columns like val_nll_uncal, test_ece_cal, etc.
    # -------------------------
    if len(df) == 1:
        d = df.iloc[0].to_dict()
        return _extract_from_flat_dict(d)

    # -------------------------
    # Case B: long format
    # possible columns:
    # split / dataset_part / phase
    # calibration / status / setting
    # metric / measure / name
    # value
    # -------------------------
    possible_split_cols = ["split", "dataset", "dataset_split", "dataset_part", "phase", "partition"]
    possible_cal_cols = ["calibration", "status", "setting", "mode"]
    possible_metric_cols = ["metric", "measure", "name"]
    possible_value_cols = ["value", "score", "metric_value"]

    split_col = next((c for c in possible_split_cols if c in df.columns), None)
    cal_col = next((c for c in possible_cal_cols if c in df.columns), None)
    metric_col = next((c for c in possible_metric_cols if c in df.columns), None)
    value_col = next((c for c in possible_value_cols if c in df.columns), None)

    if split_col and cal_col and metric_col and value_col:
        for _, r in df.iterrows():
            split = _split_from_text(r[split_col])
            cal_state = _cal_from_text(r[cal_col])
            metric = _metric_from_text(r[metric_col])
            val = _safe_float(r[value_col])
            if split and cal_state and metric and metric != "Temperature" and not pd.isna(val):
                col = f"{split} {metric} {cal_state}"
                if col in row:
                    row[col] = val
        return row

    # -------------------------
    # Case C: row-wise table with split/calibration columns and metric columns
    # -------------------------
    metric_candidates = ["nll", "brier", "ece", "mce", "temperature", "temp", "best_temperature", "optimal_temperature"]
    metric_cols_present = [c for c in df.columns if any(mc == c or mc in c for mc in metric_candidates)]

    if len(metric_cols_present) > 0:
        for _, r in df.iterrows():
            row_split = None
            row_cal = None

            for c in df.columns:
                if row_split is None:
                    row_split = _split_from_text(r[c])
                if row_cal is None:
                    row_cal = _cal_from_text(r[c])

            for c in metric_cols_present:
                metric = _metric_from_text(c)
                val = _safe_float(r[c])

                if metric == "Temperature" and not pd.isna(val):
                    row["Temperature"] = val

                if row_split and row_cal and metric and metric != "Temperature" and not pd.isna(val):
                    col = f"{row_split} {metric} {row_cal}"
                    if col in row:
                        row[col] = val
        return row

    # -------------------------
    # Fallback: flatten all rows
    # -------------------------
    for i, r in df.iterrows():
        d = {f"row{i}_{k}": v for k, v in r.to_dict().items()}
        tmp = _extract_from_flat_dict(d)
        for k, v in tmp.items():
            if pd.isna(row[k]) and not pd.isna(v):
                row[k] = v

    return row

def _extract_from_json(obj):
    row = _empty_calibration_row()

    if isinstance(obj, dict):
        flat_d = _flatten_dict(obj)
        row = _extract_from_flat_dict(flat_d)

        # also handle possible row-like lists nested inside json
        for k, v in obj.items():
            if isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict):
                try:
                    df = pd.DataFrame(v)
                    tmp = _extract_from_dataframe(df)
                    for kk, vv in tmp.items():
                        if pd.isna(row[kk]) and not pd.isna(vv):
                            row[kk] = vv
                except Exception:
                    pass

    elif isinstance(obj, list) and len(obj) > 0 and isinstance(obj[0], dict):
        df = pd.DataFrame(obj)
        row = _extract_from_dataframe(df)

    return row

def _read_any_calibration_summary(run_dir: Path):
    """
    Searches many possible files and returns:
    row_dict, source_file
    """
    candidates = []

    # exact preferred names
    exact_names = [
        "calibration_summary.csv",
        "calibration_summary.json",
        "summary.csv",
        "summary.json",
        "calibration_metrics.csv",
        "calibration_metrics.json",
        "metrics_summary.csv",
        "metrics_summary.json",
        "results.csv",
        "results.json",
        "temperature.json",
    ]
    for name in exact_names:
        p = run_dir / name
        if p.exists():
            candidates.append(p)

    # any csv/json with useful words
    for p in sorted(run_dir.iterdir()):
        if not p.is_file():
            continue
        if p.suffix.lower() not in [".csv", ".json"]:
            continue
        n = p.name.lower()
        if any(x in n for x in ["calib", "metric", "summary", "result", "temperature"]):
            if p not in candidates:
                candidates.append(p)

    best_row = _empty_calibration_row()
    best_source = None
    best_non_nan = -1

    for p in candidates:
        try:
            if p.suffix.lower() == ".json":
                obj = safe_read_json(p)
                row = _extract_from_json(obj)
            else:
                if p.stat().st_size == 0:
                    continue
                df = pd.read_csv(p)
                if len(df) == 0 and len(df.columns) == 0:
                    continue
                row = _extract_from_dataframe(df)

            non_nan_count = sum([0 if pd.isna(v) else 1 for v in row.values()])
            if non_nan_count > best_non_nan:
                best_non_nan = non_nan_count
                best_row = row
                best_source = p

        except Exception:
            continue

    return best_row, best_source, best_non_nan

# ---------- build final calibration table ----------
cal_rows = []
diag_rows = []

for run_dir in sorted(CALIBRATION_DIR.iterdir()):
    if not run_dir.is_dir():
        continue

    info = _parse_run_name_local(run_dir.name)
    extracted_row, source_file, non_nan_count = _read_any_calibration_summary(run_dir)

    final_row = {
        "Model": info["model"],
        "Seed": info["seed"],
        "Temperature": extracted_row["Temperature"],
        "Val NLL uncal": extracted_row["Val NLL uncal"],
        "Val NLL cal": extracted_row["Val NLL cal"],
        "Val Brier uncal": extracted_row["Val Brier uncal"],
        "Val Brier cal": extracted_row["Val Brier cal"],
        "Val ECE uncal": extracted_row["Val ECE uncal"],
        "Val ECE cal": extracted_row["Val ECE cal"],
        "Val MCE uncal": extracted_row["Val MCE uncal"],
        "Val MCE cal": extracted_row["Val MCE cal"],
        "Test NLL uncal": extracted_row["Test NLL uncal"],
        "Test NLL cal": extracted_row["Test NLL cal"],
        "Test Brier uncal": extracted_row["Test Brier uncal"],
        "Test Brier cal": extracted_row["Test Brier cal"],
        "Test ECE uncal": extracted_row["Test ECE uncal"],
        "Test ECE cal": extracted_row["Test ECE cal"],
        "Test MCE uncal": extracted_row["Test MCE uncal"],
        "Test MCE cal": extracted_row["Test MCE cal"],
    }

    cal_rows.append(final_row)

    diag_rows.append({
        "RunName": run_dir.name,
        "Model": info["model"],
        "Seed": info["seed"],
        "SourceFileUsed": str(source_file) if source_file is not None else "NOT_FOUND",
        "NonNaNFieldsFound": non_nan_count,
    })

cal_df = pd.DataFrame(cal_rows).sort_values(by=["Model", "Seed"]).reset_index(drop=True)
diag_df = pd.DataFrame(diag_rows).sort_values(by=["Model", "Seed"]).reset_index(drop=True)

cal_csv = TABLES_DIR / "calibration_final_table.csv"
cal_xlsx = TABLES_DIR / "calibration_final_table.xlsx"
cal_diag_csv = TABLES_DIR / "calibration_parse_diagnostics.csv"

cal_df.to_csv(cal_csv, index=False)
cal_df.to_excel(cal_xlsx, index=False)
diag_df.to_csv(cal_diag_csv, index=False)

print("Saved:", cal_csv)
print("Saved:", cal_xlsx)
print("Saved:", cal_diag_csv)

display(cal_df)
display(diag_df)

Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/calibration_final_table.csv
Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/calibration_final_table.xlsx
Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/calibration_parse_diagnostics.csv


,Model,Seed,Temperature,Val NLL uncal,Val NLL cal,Val Brier uncal,Val Brier cal,Val ECE uncal,Val ECE cal,Val MCE uncal,Val MCE cal,Test NLL uncal,Test NLL cal,Test Brier uncal,Test Brier cal,Test ECE uncal,Test ECE cal,Test MCE uncal,Test MCE cal
0,MobileNetV3-Small,13.0,0.854852,0.552178,0.524520,0.243456,0.234185,0.087534,0.074043,0.252104,0.232604,0.463741,0.424178,0.197616,0.186479,0.115854,0.058320,0.404556,0.371561
1,MobileNetV3-Small,42.0,0.882529,0.617443,0.599231,0.282419,0.276886,0.078246,0.056828,0.808987,0.798703,0.558671,0.521033,0.243274,0.230820,0.116475,0.083966,0.261537,0.241654
2,MobileNetV3-Small,87.0,0.823034,0.521429,0.480120,0.226076,0.211652,0.123952,0.074109,0.446019,0.303335,0.579798,0.546039,0.247845,0.233032,0.109546,0.064090,0.303683,0.144753
3,ResNet-18,13.0,0.812347,0.515476,0.468906,0.205951,0.185498,0.134386,0.075354,0.364873,0.301785,0.461772,0.410592,0.182876,0.164785,0.113922,0.049317,0.261314,0.119331
4,ResNet-18,42.0,0.882186,0.975303,0.957580,0.427665,0.417282,0.116639,0.085984,0.345204,0.271326,0.952841,0.933131,0.415198,0.403155,0.124220,0.092393,0.293341,0.260992
5,ResNet-18,87.0,0.783772,0.451587,0.387006,0.185558,0.165088,0.136873,0.058196,0.377698,0.624836,0.527229,0.483654,0.199619,0.179542,0.131054,0.059499,0.292002,0.304309
6,summaries,NaN,0.783772,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,RunName,Model,Seed,SourceFileUsed,NonNaNFieldsFound
0,mobilenetv3_small__adamw__weightedCE_True__lab...,MobileNetV3-Small,13.0,/mnt/e/CVPR/steel_failure_aware/runs/calibrati...,17
1,mobilenetv3_small__adamw__weightedCE_True__lab...,MobileNetV3-Small,42.0,/mnt/e/CVPR/steel_failure_aware/runs/calibrati...,17
2,mobilenetv3_small__adamw__weightedCE_True__lab...,MobileNetV3-Small,87.0,/mnt/e/CVPR/steel_failure_aware/runs/calibrati...,17
3,resnet18__adamw__weightedCE_True__labelsmooth_...,ResNet-18,13.0,/mnt/e/CVPR/steel_failure_aware/runs/calibrati...,17
4,resnet18__adamw__weightedCE_True__labelsmooth_...,ResNet-18,42.0,/mnt/e/CVPR/steel_failure_aware/runs/calibrati...,17
5,resnet18__adamw__weightedCE_True__labelsmooth_...,ResNet-18,87.0,/mnt/e/CVPR/steel_failure_aware/runs/calibrati...,17
6,summaries,summaries,NaN,/mnt/e/CVPR/steel_failure_aware/runs/calibrati...,1


In [12]:
selective_rows = []
fixed_rows = []

for run_dir in SELECTIVE_DIR.iterdir():
    if not run_dir.is_dir():
        continue

    info = parse_run_name(run_dir.name)

    # ---------- summary ----------
    summary_candidates = [
        run_dir / "selective_prediction_summary.json",
        run_dir / "selective_prediction_summary.csv",
        run_dir / "summary.json",
        run_dir / "summary.csv",
    ]
    summary_file = first_existing_file(summary_candidates)

    if summary_file is not None:
        if summary_file.suffix.lower() == ".json":
            summary_obj = safe_read_json(summary_file)
        else:
            summary_obj = pd.read_csv(summary_file)

        # Case A: JSON dict keyed by score names
        if isinstance(summary_obj, dict):
            for score_name, sd in summary_obj.items():
                if isinstance(sd, dict) and score_name in ["raw_msp", "calibrated_msp", "entropy", "max_logit"]:
                    selective_rows.append({
                        "Model": info["model"],
                        "Seed": info["seed"],
                        "Score": score_name,
                        "AURC": sd.get("aurc", sd.get("AURC", np.nan)),
                        "E-AURC": sd.get("eaurc", sd.get("E-AURC", np.nan)),
                        "Full coverage accuracy": sd.get("full_coverage_accuracy", sd.get("accuracy_full_coverage", np.nan)),
                        "Full coverage risk": sd.get("full_coverage_risk", sd.get("risk_full_coverage", np.nan)),
                    })

        # Case B: CSV or DataFrame already row-wise
        elif isinstance(summary_obj, pd.DataFrame):
            sdf = summary_obj.copy()
            if "score_name" in sdf.columns:
                for _, r in sdf.iterrows():
                    selective_rows.append({
                        "Model": info["model"],
                        "Seed": info["seed"],
                        "Score": r.get("score_name", np.nan),
                        "AURC": r.get("aurc", r.get("AURC", np.nan)),
                        "E-AURC": r.get("eaurc", r.get("E-AURC", np.nan)),
                        "Full coverage accuracy": r.get("full_coverage_accuracy", r.get("accuracy_full_coverage", np.nan)),
                        "Full coverage risk": r.get("full_coverage_risk", r.get("risk_full_coverage", np.nan)),
                    })

    # ---------- fixed coverage ----------
    for score_name in ["raw_msp", "calibrated_msp", "entropy", "max_logit"]:
        fc_path = run_dir / f"selective_metrics_fixed_coverage_{score_name}.csv"
        if not fc_path.exists():
            continue

        fc_df = pd.read_csv(fc_path)
        fc_df["Model"] = info["model"]
        fc_df["Seed"] = info["seed"]
        fc_df["Score"] = score_name
        fixed_rows.append(fc_df)

selective_df = pd.DataFrame(selective_rows)
if len(selective_df) > 0:
    selective_df = selective_df.sort_values(by=["Model", "Seed", "AURC"], ascending=[True, True, True]).reset_index(drop=True)

selective_csv = TABLES_DIR / "selective_prediction_benchmark_table.csv"
selective_xlsx = TABLES_DIR / "selective_prediction_benchmark_table.xlsx"
selective_df.to_csv(selective_csv, index=False)
selective_df.to_excel(selective_xlsx, index=False)

print("Saved:", selective_csv)
print("Saved:", selective_xlsx)
display(selective_df)

if len(fixed_rows) > 0:
    selective_fixed_df = pd.concat(fixed_rows, ignore_index=True)

    selective_fixed_csv = TABLES_DIR / "selective_prediction_fixed_coverage_table.csv"
    selective_fixed_xlsx = TABLES_DIR / "selective_prediction_fixed_coverage_table.xlsx"

    selective_fixed_df.to_csv(selective_fixed_csv, index=False)
    selective_fixed_df.to_excel(selective_fixed_xlsx, index=False)

    print("Saved:", selective_fixed_csv)
    print("Saved:", selective_fixed_xlsx)
    display(selective_fixed_df.head(20))
else:
    print("No fixed coverage files found.")

Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/selective_prediction_benchmark_table.csv
Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/selective_prediction_benchmark_table.xlsx


""


Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/selective_prediction_fixed_coverage_table.csv
Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/selective_prediction_fixed_coverage_table.xlsx


,coverage,n_selected,selective_accuracy,selective_macro_f1,Model,Seed,Score
0,1.00,347,0.876081,0.787633,MobileNetV3-Small,13,raw_msp
1,0.95,330,0.909091,0.832267,MobileNetV3-Small,13,raw_msp
2,0.90,313,0.916933,0.848980,MobileNetV3-Small,13,raw_msp
3,0.80,278,0.949640,0.907434,MobileNetV3-Small,13,raw_msp
4,0.70,243,0.971193,0.928224,MobileNetV3-Small,13,raw_msp
5,0.60,209,0.985646,0.954702,MobileNetV3-Small,13,raw_msp
6,0.50,174,0.982759,0.953389,MobileNetV3-Small,13,raw_msp
7,1.00,347,0.876081,0.787633,MobileNetV3-Small,13,calibrated_msp
8,0.95,330,0.909091,0.832267,MobileNetV3-Small,13,calibrated_msp
9,0.90,313,0.916933,0.846094,MobileNetV3-Small,13,calibrated_msp


In [6]:
neu_summary_candidates = [
    EXTERNAL_NEU_DIR / "summaries" / "final_neu_external_unknown_table.csv",
    EXTERNAL_NEU_DIR / "final_neu_external_unknown_table.csv",
]

neu_final_file = first_existing_file(neu_summary_candidates)

if neu_final_file is not None:
    neu_df = pd.read_csv(neu_final_file)
else:
    neu_rows = []

    for run_dir in EXTERNAL_NEU_DIR.iterdir():
        if not run_dir.is_dir():
            continue

        info = parse_run_name(run_dir.name)

        ood_csv = run_dir / "ood_metrics.csv"
        thr_csv = run_dir / "threshold_rejection_table.csv"

        if not ood_csv.exists() or not thr_csv.exists():
            continue

        ood_df = pd.read_csv(ood_csv)
        thr_df = pd.read_csv(thr_csv)

        if len(ood_df) == 0 or len(thr_df) == 0:
            continue

        o = ood_df.iloc[0]

        row = {
            "Model": info["model"],
            "Seed": info["seed"],
            "Score": info["score"],
            "AUROC": o.get("auroc_unknown_positive", np.nan),
            "AUPR": o.get("aupr_unknown_positive", np.nan),
            "FPR@95TPR": o.get("fpr_at_95_tpr", np.nan),
        }

        for cov in [0.95, 0.90, 0.80]:
            sub = thr_df[np.isclose(thr_df["target_val_coverage"], cov)]
            if len(sub) == 0:
                continue
            s = sub.iloc[0]
            tag = int(cov * 100)
            row[f"Known accept rate @ {tag}"] = s.get("known_accept_rate_test", np.nan)
            row[f"Known acc after reject @ {tag}"] = s.get("known_accuracy_after_reject", np.nan)
            row[f"Unknown reject rate @ {tag}"] = s.get("unknown_reject_rate_neu", s.get("unknown_reject_rate_unknown", np.nan))

        neu_rows.append(row)

    neu_df = pd.DataFrame(neu_rows)

neu_df = neu_df.sort_values(by=["AUROC", "AUPR"], ascending=[False, False]).reset_index(drop=True)

# highlight requested rows
neu_df["HighlightForMainText"] = (
    ((neu_df["Model"] == "MobileNetV3-Small") & (neu_df["Score"] == "raw_msp") & (neu_df["Seed"] == 13)) |
    ((neu_df["Model"] == "MobileNetV3-Small") & (neu_df["Score"] == "raw_msp") & (neu_df["Seed"] == 87))
)

neu_csv = TABLES_DIR / "external_neu_unknown_final_table.csv"
neu_xlsx = TABLES_DIR / "external_neu_unknown_final_table.xlsx"
neu_df.to_csv(neu_csv, index=False)
neu_df.to_excel(neu_xlsx, index=False)

print("Saved:", neu_csv)
print("Saved:", neu_xlsx)
display(neu_df)

Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/external_neu_unknown_final_table.csv
Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/external_neu_unknown_final_table.xlsx


,Model,Seed,Score,AUROC,AUPR,FPR@95TPR,Known accept rate @ 95,Known acc after reject @ 95,Unknown reject rate @ 95,Known accept rate @ 90,Known acc after reject @ 90,Unknown reject rate @ 90,Known accept rate @ 80,Known acc after reject @ 80,Unknown reject rate @ 80,HighlightForMainText
0,resnet18,87,raw_msp,0.959978,0.988695,0.138329,0.951009,0.915152,0.753333,0.913545,0.924290,0.881111,0.801153,0.942446,0.978889,False
1,resnet18,87,calibrated_msp,0.957880,0.988317,0.149856,0.953890,0.915408,0.729444,0.910663,0.924051,0.856667,0.798271,0.945848,0.980556,False
2,resnet18,13,raw_msp,0.915623,0.979211,0.317003,0.968300,0.904762,0.466667,0.907781,0.933333,0.716111,0.806916,0.964286,0.886667,False
3,resnet18,13,calibrated_msp,0.914299,0.978441,0.302594,0.971182,0.902077,0.446111,0.910663,0.930380,0.693333,0.809798,0.964413,0.893333,False
4,mobilenetv3_small,13,raw_msp,0.862967,0.965660,0.512968,0.956772,0.903614,0.395556,0.913545,0.914826,0.593333,0.827089,0.944251,0.736667,False
5,mobilenetv3_small,13,calibrated_msp,0.862099,0.964969,0.487032,0.951009,0.909091,0.411667,0.902017,0.916933,0.586111,0.812680,0.950355,0.746111,False
6,mobilenetv3_small,87,raw_msp,0.860889,0.967934,0.659942,0.942363,0.880734,0.581667,0.893372,0.893548,0.667222,0.763689,0.932075,0.796667,False
7,mobilenetv3_small,87,calibrated_msp,0.856817,0.966463,0.654179,0.936599,0.876923,0.541667,0.902017,0.894569,0.626111,0.775216,0.929368,0.781667,False
8,mobilenetv3_small,42,raw_msp,0.783565,0.953436,0.991354,0.930836,0.876161,0.552222,0.873199,0.900990,0.658889,0.752161,0.946360,0.748333,False
9,mobilenetv3_small,42,calibrated_msp,0.781417,0.952672,0.991354,0.936599,0.873846,0.542778,0.878963,0.901639,0.640556,0.752161,0.950192,0.745556,False


In [7]:
loco_summary_csv = LOCO_DIR / "loco_summary_across_folds.csv"
loco_threshold_csv = LOCO_DIR / "loco_threshold_summary_across_folds.csv"
all_loco_metrics_csv = LOCO_DIR / "all_loco_metrics.csv"
all_loco_thresholds_csv = LOCO_DIR / "all_loco_thresholds.csv"

if not loco_summary_csv.exists():
    raise FileNotFoundError(f"Missing: {loco_summary_csv}")
if not loco_threshold_csv.exists():
    raise FileNotFoundError(f"Missing: {loco_threshold_csv}")

loco_summary_df = pd.read_csv(loco_summary_csv)
loco_threshold_df = pd.read_csv(loco_threshold_csv)

loco_summary_out_csv = TABLES_DIR / "internal_gc10_loco_summary_table.csv"
loco_summary_out_xlsx = TABLES_DIR / "internal_gc10_loco_summary_table.xlsx"
loco_summary_df.to_csv(loco_summary_out_csv, index=False)
loco_summary_df.to_excel(loco_summary_out_xlsx, index=False)

loco_threshold_out_csv = TABLES_DIR / "internal_gc10_loco_threshold_table.csv"
loco_threshold_out_xlsx = TABLES_DIR / "internal_gc10_loco_threshold_table.xlsx"
loco_threshold_df.to_csv(loco_threshold_out_csv, index=False)
loco_threshold_df.to_excel(loco_threshold_out_xlsx, index=False)

print("Saved:", loco_summary_out_csv)
print("Saved:", loco_summary_out_xlsx)
print("Saved:", loco_threshold_out_csv)
print("Saved:", loco_threshold_out_xlsx)

display(loco_summary_df)
display(loco_threshold_df)

if all_loco_metrics_csv.exists():
    all_loco_metrics_df = pd.read_csv(all_loco_metrics_csv)
    all_loco_metrics_out = TABLES_DIR / "internal_gc10_loco_all_fold_metrics.csv"
    all_loco_metrics_df.to_csv(all_loco_metrics_out, index=False)
    print("Saved:", all_loco_metrics_out)

if all_loco_thresholds_csv.exists():
    all_loco_thresholds_df = pd.read_csv(all_loco_thresholds_csv)
    all_loco_thresholds_out = TABLES_DIR / "internal_gc10_loco_all_fold_thresholds.csv"
    all_loco_thresholds_df.to_csv(all_loco_thresholds_out, index=False)
    print("Saved:", all_loco_thresholds_out)

Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/internal_gc10_loco_summary_table.csv
Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/internal_gc10_loco_summary_table.xlsx
Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/internal_gc10_loco_threshold_table.csv
Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/internal_gc10_loco_threshold_table.xlsx


,model,seed,score_name,mean_auroc,std_auroc,mean_aupr,std_aupr,mean_fpr95,std_fpr95
0,mobilenetv3_small,42,raw_msp,0.683701,0.100603,0.561186,0.186034,0.848189,0.127947
1,mobilenetv3_small,42,calibrated_msp,0.684971,0.095882,0.560400,0.180769,0.843348,0.133899


,model,seed,score_name,target_val_coverage,mean_known_accept_rate_test,mean_known_accuracy_after_reject,mean_unknown_reject_rate
0,mobilenetv3_small,42,raw_msp,0.95,0.951959,0.866132,0.189476
1,mobilenetv3_small,42,raw_msp,0.90,0.908379,0.879066,0.308626
2,mobilenetv3_small,42,raw_msp,0.80,0.815096,0.900033,0.470720
3,mobilenetv3_small,42,calibrated_msp,0.95,0.950141,0.867700,0.183460
4,mobilenetv3_small,42,calibrated_msp,0.90,0.910875,0.880985,0.291052
5,mobilenetv3_small,42,calibrated_msp,0.80,0.811054,0.908167,0.468215


Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/internal_gc10_loco_all_fold_metrics.csv
Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/internal_gc10_loco_all_fold_thresholds.csv


In [8]:
def find_by_patterns(folder: Path, patterns):
    matches = []
    for pat in patterns:
        matches.extend(list(folder.glob(pat)))
    matches = [m for m in matches if m.exists()]
    if len(matches) == 0:
        return None
    matches = sorted(matches)
    return matches[0]

figure_manifest = []

# ---------------------------------
# 1) Closed-set key confusion matrices
# ---------------------------------
closed_fig_targets = [
    (
        CLOSED_SET_DIR / "mobilenetv3_small__adamw__weightedCE_True__labelsmooth_True__pretrained_True__seed_42",
        ["confusion_matrix.png"],
        FIGURES_DIR / "figure01_closedset_confusion_mobilenet_seed42.png"
    ),
    (
        CLOSED_SET_DIR / "resnet18__adamw__weightedCE_True__labelsmooth_True__pretrained_True__seed_42",
        ["confusion_matrix.png"],
        FIGURES_DIR / "figure02_closedset_confusion_resnet_seed42.png"
    ),
]

for src_dir, patterns, dst in closed_fig_targets:
    src = find_by_patterns(src_dir, patterns)
    ok = copy_if_exists(src, dst)
    figure_manifest.append({
        "figure_name": dst.name,
        "copied": ok,
        "source": str(src) if src is not None else "NOT_FOUND"
    })

# ---------------------------------
# 2) Calibration reliability diagrams
# ---------------------------------
cal_fig_targets = [
    (
        CALIBRATION_DIR / "mobilenetv3_small__adamw__weightedCE_True__labelsmooth_True__pretrained_True__seed_42",
        ["*test*calibrated*.png", "*Test*Calibrated*.png", "*reliability*calibrated*.png"],
        FIGURES_DIR / "figure03_calibration_test_reliability_mobilenet_seed42_calibrated.png"
    ),
    (
        CALIBRATION_DIR / "resnet18__adamw__weightedCE_True__labelsmooth_True__pretrained_True__seed_42",
        ["*test*calibrated*.png", "*Test*Calibrated*.png", "*reliability*calibrated*.png"],
        FIGURES_DIR / "figure04_calibration_test_reliability_resnet_seed42_calibrated.png"
    ),
]

for src_dir, patterns, dst in cal_fig_targets:
    src = find_by_patterns(src_dir, patterns)
    ok = copy_if_exists(src, dst)
    figure_manifest.append({
        "figure_name": dst.name,
        "copied": ok,
        "source": str(src) if src is not None else "NOT_FOUND"
    })

# ---------------------------------
# 3) Selective prediction figure
# ---------------------------------
selective_fig_targets = [
    (
        SELECTIVE_DIR / "mobilenetv3_small__adamw__weightedCE_True__labelsmooth_True__pretrained_True__seed_42",
        ["risk_coverage_all_scores.png"],
        FIGURES_DIR / "figure05_selective_risk_coverage_mobilenet_seed42.png"
    ),
    (
        SELECTIVE_DIR / "mobilenetv3_small__adamw__weightedCE_True__labelsmooth_True__pretrained_True__seed_42",
        ["selective_macro_f1_fixed_coverages.png"],
        FIGURES_DIR / "figure06_selective_macro_f1_mobilenet_seed42.png"
    ),
]

for src_dir, patterns, dst in selective_fig_targets:
    src = find_by_patterns(src_dir, patterns)
    ok = copy_if_exists(src, dst)
    figure_manifest.append({
        "figure_name": dst.name,
        "copied": ok,
        "source": str(src) if src is not None else "NOT_FOUND"
    })

# ---------------------------------
# 4) External NEU histogram figures
# ---------------------------------
external_fig_targets = [
    (
        EXTERNAL_NEU_DIR / "mobilenetv3_small__adamw__weightedCE_True__labelsmooth_True__pretrained_True__seed_13__score_raw_msp",
        ["known_vs_unknown_score_histogram.png"],
        FIGURES_DIR / "figure07_external_neu_hist_mobilenet_seed13_raw_msp.png"
    ),
    (
        EXTERNAL_NEU_DIR / "mobilenetv3_small__adamw__weightedCE_True__labelsmooth_True__pretrained_True__seed_87__score_raw_msp",
        ["known_vs_unknown_score_histogram.png"],
        FIGURES_DIR / "figure08_external_neu_hist_mobilenet_seed87_raw_msp.png"
    ),
]

for src_dir, patterns, dst in external_fig_targets:
    src = find_by_patterns(src_dir, patterns)
    ok = copy_if_exists(src, dst)
    figure_manifest.append({
        "figure_name": dst.name,
        "copied": ok,
        "source": str(src) if src is not None else "NOT_FOUND"
    })

# ---------------------------------
# 5) Internal LOCO figures
# ---------------------------------
loco_fig_targets = [
    (
        LOCO_DIR,
        ["loco_auroc_by_fold.png"],
        FIGURES_DIR / "figure09_internal_loco_auroc_by_fold.png"
    ),
    (
        LOCO_DIR,
        ["loco_fpr95_by_fold.png"],
        FIGURES_DIR / "figure10_internal_loco_fpr95_by_fold.png"
    ),
]

for src_dir, patterns, dst in loco_fig_targets:
    src = find_by_patterns(src_dir, patterns)
    ok = copy_if_exists(src, dst)
    figure_manifest.append({
        "figure_name": dst.name,
        "copied": ok,
        "source": str(src) if src is not None else "NOT_FOUND"
    })

figure_manifest_df = pd.DataFrame(figure_manifest)
manifest_csv = FIGURES_DIR / "final_figure_manifest.csv"
figure_manifest_df.to_csv(manifest_csv, index=False)

print("Saved:", manifest_csv)
display(figure_manifest_df)

Saved: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/figures/final_figure_manifest.csv


,figure_name,copied,source
0,figure01_closedset_confusion_mobilenet_seed42.png,True,/mnt/e/CVPR/steel_failure_aware/runs/closed_se...
1,figure02_closedset_confusion_resnet_seed42.png,True,/mnt/e/CVPR/steel_failure_aware/runs/closed_se...
2,figure03_calibration_test_reliability_mobilene...,True,/mnt/e/CVPR/steel_failure_aware/runs/calibrati...
3,figure04_calibration_test_reliability_resnet_s...,True,/mnt/e/CVPR/steel_failure_aware/runs/calibrati...
4,figure05_selective_risk_coverage_mobilenet_see...,True,/mnt/e/CVPR/steel_failure_aware/runs/selective...
5,figure06_selective_macro_f1_mobilenet_seed42.png,True,/mnt/e/CVPR/steel_failure_aware/runs/selective...
6,figure07_external_neu_hist_mobilenet_seed13_ra...,True,/mnt/e/CVPR/steel_failure_aware/runs/external_...
7,figure08_external_neu_hist_mobilenet_seed87_ra...,True,/mnt/e/CVPR/steel_failure_aware/runs/external_...
8,figure09_internal_loco_auroc_by_fold.png,True,/mnt/e/CVPR/steel_failure_aware/runs/internal_...
9,figure10_internal_loco_fpr95_by_fold.png,True,/mnt/e/CVPR/steel_failure_aware/runs/internal_...


In [13]:
# =========================
# Cell 9 (REPLACEMENT)
# Robust workbook writer
# Skips missing or empty CSVs
# =========================
import pandas as pd
from pandas.errors import EmptyDataError

excel_path = FINAL_DIR / "final_tables_workbook.xlsx"

table_files = {
    "closed_set_per_run": TABLES_DIR / "closed_set_final_benchmark_table.csv",
    "closed_set_grouped": TABLES_DIR / "closed_set_grouped_mean_std_table.csv",
    "calibration": TABLES_DIR / "calibration_final_table.csv",
    "selective_benchmark": TABLES_DIR / "selective_prediction_benchmark_table.csv",
    "selective_fixed": TABLES_DIR / "selective_prediction_fixed_coverage_table.csv",
    "external_neu": TABLES_DIR / "external_neu_unknown_final_table.csv",
    "loco_summary": TABLES_DIR / "internal_gc10_loco_summary_table.csv",
    "loco_thresholds": TABLES_DIR / "internal_gc10_loco_threshold_table.csv",
}

sheet_log = []

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, path in table_files.items():
        if not path.exists():
            print(f"Skipping missing file: {path}")
            sheet_log.append({"sheet": sheet_name, "path": str(path), "status": "missing"})
            continue

        if path.stat().st_size == 0:
            print(f"Skipping empty file (0 bytes): {path}")
            sheet_log.append({"sheet": sheet_name, "path": str(path), "status": "empty_0_bytes"})
            continue

        try:
            df = pd.read_csv(path)

            # skip fully empty tables
            if len(df.columns) == 0:
                print(f"Skipping empty file (no columns): {path}")
                sheet_log.append({"sheet": sheet_name, "path": str(path), "status": "empty_no_columns"})
                continue

            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
            print(f"Added sheet: {sheet_name}")
            sheet_log.append({"sheet": sheet_name, "path": str(path), "status": "written", "rows": len(df), "cols": len(df.columns)})

        except EmptyDataError:
            print(f"Skipping unreadable empty CSV: {path}")
            sheet_log.append({"sheet": sheet_name, "path": str(path), "status": "emptydataerror"})
        except Exception as e:
            print(f"Skipping {path} بسبب error: {e}")
            sheet_log.append({"sheet": sheet_name, "path": str(path), "status": f"error: {e}"})

sheet_log_df = pd.DataFrame(sheet_log)
sheet_log_csv = FINAL_DIR / "final_tables_workbook_log.csv"
sheet_log_df.to_csv(sheet_log_csv, index=False)

print("Saved workbook:", excel_path)
print("Saved log:", sheet_log_csv)

display(sheet_log_df)

Added sheet: closed_set_per_run
Added sheet: closed_set_grouped
Added sheet: calibration
Skipping unreadable empty CSV: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/tables/selective_prediction_benchmark_table.csv
Added sheet: selective_fixed
Added sheet: external_neu
Added sheet: loco_summary
Added sheet: loco_thresholds
Saved workbook: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/final_tables_workbook.xlsx
Saved log: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/final_tables_workbook_log.csv


,sheet,path,status,rows,cols
0,closed_set_per_run,/mnt/e/CVPR/steel_failure_aware/final_tables_a...,written,10.0,36.0
1,closed_set_grouped,/mnt/e/CVPR/steel_failure_aware/final_tables_a...,written,5.0,10.0
2,calibration,/mnt/e/CVPR/steel_failure_aware/final_tables_a...,written,7.0,19.0
3,selective_benchmark,/mnt/e/CVPR/steel_failure_aware/final_tables_a...,emptydataerror,NaN,NaN
4,selective_fixed,/mnt/e/CVPR/steel_failure_aware/final_tables_a...,written,168.0,7.0
5,external_neu,/mnt/e/CVPR/steel_failure_aware/final_tables_a...,written,12.0,16.0
6,loco_summary,/mnt/e/CVPR/steel_failure_aware/final_tables_a...,written,2.0,9.0
7,loco_thresholds,/mnt/e/CVPR/steel_failure_aware/final_tables_a...,written,6.0,7.0


In [10]:
main_text_dir = FINAL_DIR / "writing_notes"
main_text_dir.mkdir(parents=True, exist_ok=True)

discussion_text = """NEU represents a real cross-dataset shift relative to GC10, so a noticeable overlap between known and unknown confidence distributions is expected. The external unknown rejection results show that rejection is useful, but still far from solved. In our experiments, calibration improved selective prediction inside the closed-set setting more consistently than it improved external unknown rejection on NEU. This indicates that confidence refinement helps abstention on in-distribution hard cases, but cross-dataset unknown detection remains a more difficult problem. Even so, the lightweight MobileNetV3-Small model still provides a practical rejection mechanism, showing that a compact industrial model can retain strong closed-set performance while offering meaningful unknown filtering under realistic dataset shift. Overall, the NEU evaluation confirms that the external unknown setting is substantially harder than the closed-set selective setting, and should be interpreted as a stress test rather than a solved rejection scenario.
"""

highlight_text = """For the NEU external-unknown experiment, we highlight MobileNetV3-Small with raw MSP at seed 13 as the strongest verified run, with MobileNetV3-Small with raw MSP at seed 87 as the secondary result.
"""

loco_text = """The GC10 leave-one-class-out internal unknown experiment showed that internal unknown-defect rejection remains challenging in this steel surface setting. Averaged across the 10 held-out classes, both raw MSP and calibrated MSP achieved only moderate AUROC and high FPR@95TPR, indicating substantial overlap between known and held-out defect classes. Temperature-scaled MSP provided only a marginal average improvement over raw MSP, with slightly better AUROC and slightly lower FPR@95TPR, while the largest practical gain appeared in accepted-known accuracy at fixed coverage rather than in unknown rejection itself. These results suggest that some GC10 defect categories are visually close enough that holding one class out does not create an easy unknown-detection problem. Therefore, the internal unknown setting should be interpreted as a hard semantic rejection test rather than a simple within-dataset novelty benchmark.
"""

with open(main_text_dir / "external_neu_discussion.txt", "w", encoding="utf-8") as f:
    f.write(discussion_text)

with open(main_text_dir / "external_neu_highlight.txt", "w", encoding="utf-8") as f:
    f.write(highlight_text)

with open(main_text_dir / "internal_loco_discussion.txt", "w", encoding="utf-8") as f:
    f.write(loco_text)

print("Saved writing notes to:", main_text_dir)

Saved writing notes to: /mnt/e/CVPR/steel_failure_aware/final_tables_and_figures/writing_notes
